# Probar CBR Creative Intelligence

Notebook simple para construir el indice CBR, consultar vecinos similares, ver explicaciones y ejecutar una evaluacion rapida.

Si falta alguna dependencia, instala primero:

```python
%pip install -r ../requirements.txt
```

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DATA_PATH = ROOT / "data" / "creative_cbr_cases_final.parquet"
FEATURE_SETS_PATH = ROOT / "data" / "cbr_feature_sets.json"

# Fallback a la ubicacion actual del repo.
if not DATA_PATH.exists():
    DATA_PATH = ROOT / "dataset" / "final" / "creative_cbr_cases_final.parquet"
if not FEATURE_SETS_PATH.exists():
    FEATURE_SETS_PATH = ROOT / "dataset" / "final" / "cbr_feature_sets.json"

CONFIG_PATH = ROOT / "configs" / "cbr_prelaunch.yaml"
INDEX_DIR = ROOT / "outputs" / "cbr_index_notebook"

DATA_PATH, FEATURE_SETS_PATH, CONFIG_PATH, INDEX_DIR

(WindowsPath('c:/Users/sylbo/Documents_local/hack_UPC_2026/dataset/final/creative_cbr_cases_final.parquet'),
 WindowsPath('c:/Users/sylbo/Documents_local/hack_UPC_2026/dataset/final/cbr_feature_sets.json'),
 WindowsPath('c:/Users/sylbo/Documents_local/hack_UPC_2026/configs/cbr_prelaunch.yaml'),
 WindowsPath('c:/Users/sylbo/Documents_local/hack_UPC_2026/outputs/cbr_index_notebook'))

## 1. Cargar configuracion

In [2]:
from cbr_engine import CBRConfig

config = CBRConfig.from_yaml(CONFIG_PATH)
config.mode, config.feature_set_name, config.block_weights()

('prelaunch',
 'prelaunch_feature_cols',
 {'clip': 0.45,
  'cnn': 0.25,
  'visual_numeric': 0.2,
  'text_numeric': 0.05,
  'categorical_context': 0.05})

## 2. Construir el indice

Usa `force=True` para reconstruirlo. Si ya existe y no quieres recalcular, cambia a `force=False`.

In [3]:
from cbr_engine import build_cbr_index

manifest = build_cbr_index(
    data_path=str(DATA_PATH),
    feature_sets_path=str(FEATURE_SETS_PATH),
    config=config,
    output_dir=str(INDEX_DIR),
    force=True,
)

manifest

{'timestamp': '2026-04-26T01:59:18.046290+00:00',
 'num_cases': 1080,
 'vector_dim': 386,
 'blocks': ['clip',
  'cnn',
  'visual_numeric',
  'text_numeric',
  'categorical_context'],
 'weights': {'clip': 0.45,
  'cnn': 0.25,
  'visual_numeric': 0.2,
  'text_numeric': 0.05,
  'categorical_context': 0.05},
 'columns_by_block': {'clip': ['clip_pca_0',
   'clip_pca_1',
   'clip_pca_2',
   'clip_pca_3',
   'clip_pca_4',
   'clip_pca_5',
   'clip_pca_6',
   'clip_pca_7',
   'clip_pca_8',
   'clip_pca_9',
   'clip_pca_10',
   'clip_pca_11',
   'clip_pca_12',
   'clip_pca_13',
   'clip_pca_14',
   'clip_pca_15',
   'clip_pca_16',
   'clip_pca_17',
   'clip_pca_18',
   'clip_pca_19',
   'clip_pca_20',
   'clip_pca_21',
   'clip_pca_22',
   'clip_pca_23',
   'clip_pca_24',
   'clip_pca_25',
   'clip_pca_26',
   'clip_pca_27',
   'clip_pca_28',
   'clip_pca_29',
   'clip_pca_30',
   'clip_pca_31',
   'clip_pca_32',
   'clip_pca_33',
   'clip_pca_34',
   'clip_pca_35',
   'clip_pca_36',
   'clip_p

## 3. Cargar retriever

In [4]:
from cbr_engine import load_retriever

retriever = load_retriever(str(INDEX_DIR), config)
len(retriever.ids), retriever.matrix.shape

(1080, (1080, 386))

## 4. Elegir una creatividad y recuperar vecinos

In [5]:
# Cambia este creative_id por el que quieras probar.
creative_id = int(retriever.ids[0])
creative_id

500000

In [6]:
neighbors, trace = retriever.retrieve_by_id(
    creative_id,
    k=10,
    filters={
        "same_vertical": True,
        "same_format": True,
        "exclude_same_campaign": False,
    },
)

display_cols = [
    "rank",
    "neighbor_creative_id",
    "global_similarity",
    "similarity_clip",
    "similarity_cnn",
    "similarity_visual_numeric",
    "similarity_text_numeric",
    "similarity_categorical_context",
    "final_score",
    "creative_status",
    "perf_score",
    "overall_roas",
    "overall_ipm",
    "reason_codes",
]

neighbors[[c for c in display_cols if c in neighbors.columns]]

,rank,neighbor_creative_id,global_similarity,similarity_clip,similarity_cnn,similarity_visual_numeric,similarity_text_numeric,similarity_categorical_context,final_score,creative_status,perf_score,overall_roas,overall_ipm,reason_codes
0,1,500003,0.547008,0.606760,0.349160,0.659746,0.506300,0.588235,0.572072,stable,0.491667,0.949188,1.036812,"[SAME_VERTICAL, SAME_FORMAT, POSSIBLE_CAMPAIGN..."
1,2,500613,0.531027,0.368159,0.846832,0.501638,0.419336,0.647059,0.557426,fatigued,0.608333,1.198638,1.276904,"[HIGH_CNN_SIMILARITY, SAME_VERTICAL, SAME_FORMAT]"
2,3,500072,0.551803,0.851296,0.129101,0.433446,0.700986,0.294118,0.556172,stable,0.553086,1.080667,1.173628,"[HIGH_CLIP_SIMILARITY, SAME_VERTICAL, SAME_FOR..."
3,4,500144,0.507129,0.534470,0.505977,0.476999,0.365065,0.529412,0.539522,stable,0.531790,1.358610,0.971114,"[SAME_VERTICAL, SAME_FORMAT]"
4,5,500792,0.496075,0.460549,0.508205,0.533478,0.689851,0.411765,0.534672,stable,0.728704,1.726607,1.706910,"[SAME_VERTICAL, SAME_FORMAT]"
5,6,500362,0.440635,0.512905,0.238804,0.552681,0.262387,0.529412,0.512622,stable,0.540432,1.091048,1.061088,"[SAME_VERTICAL, SAME_FORMAT]"
6,7,500364,0.434544,0.162095,0.767879,0.616171,0.339707,0.588235,0.512388,stable,0.557716,1.186315,1.163691,"[HIGH_CNN_SIMILARITY, SAME_VERTICAL, SAME_FORMAT]"
7,8,500434,0.473871,0.287659,0.508928,0.877047,0.365065,0.470588,0.508361,stable,0.557716,1.232626,1.000248,"[SAME_VERTICAL, SAME_FORMAT]"
8,9,500397,0.471850,0.785421,0.014228,0.225133,0.984785,0.411765,0.493724,stable,0.561111,0.984677,1.101359,"[HIGH_CLIP_SIMILARITY, SAME_VERTICAL, SAME_FOR..."
9,10,500721,0.447519,0.543009,0.319322,0.373503,0.502092,0.470588,0.490692,fatigued,0.602778,1.250622,1.213497,"[SAME_VERTICAL, SAME_FORMAT]"


## 5. Agregados, recomendacion y explicaciones

In [7]:
enriched = retriever.enrich(neighbors)

enriched["reuse_summary"]

{'neighbor_count': 10,
 'avg_similarity': 0.49014607071876526,
 'avg_final_score': 0.5277650558668585,
 'median_perf_score': 0.5577160493827161,
 'mean_perf_score': 0.5733333333333334,
 'median_roas': 1.1924765728780122,
 'median_ipm': 1.1325251568288293,
 'median_ctr': 0.00613890615885555,
 'median_cvr': 0.1809403205018787,
 'top_performer_ratio': 0.0,
 'fatigued_ratio': 0.2,
 'stable_ratio': 0.8,
 'weighted_perf_score': 0.5729155577414443,
 'weighted_roas': 1.205452584048728,
 'weighted_ipm': 1.1713790716493033,
 'confidence_score': 0.6131923601109138}

In [8]:
enriched["recommendation"]

{'action': 'PIVOT',
 'confidence': 0.6131923601109138,
 'summary': 'PIVOT con confianza 0.61',
 'reasons': ['La similitud es razonable pero los outcomes historicos son mixtos o bajos.'],
 'risks': [],
 'suggested_next_steps': ['Cambiar hook, layout o propuesta visual antes de invertir mas.']}

In [9]:
print(enriched["explanation"]["marketer_explanation"])
print()
print(enriched["explanation"]["technical_explanation"])

Esta creatividad se parece a 10 casos historicos. La similitud media es 0.49. La similitud viene sobre todo de visual_numeric, text_numeric, clip. Los vecinos tuvieron ROAS mediano 1.1924765728780122 e IPM mediano 1.1325251568288293. El 0% fueron top performers. Recomendacion: PIVOT con confianza 0.61.

Top bloques por similitud media: [{'block': 'visual_numeric', 'mean_similarity': 0.525}, {'block': 'text_numeric', 'mean_similarity': 0.5136}, {'block': 'clip', 'mean_similarity': 0.5112}]. Patrones diferenciales detectados: []. Warnings: [].


## 6. Ver trazabilidad de la query

In [10]:
trace.keys(), trace.get("warnings")

(dict_keys(['query_creative_id', 'config', 'filters_applied', 'k_requested', 'overfetch', 'discarded_by_filter', 'warnings', 'neighbors_final']),
 [])

In [11]:
trace["config"]

{'mode': 'prelaunch',
 'feature_set_name': 'prelaunch_feature_cols',
 'weights': {'clip': 0.45,
  'cnn': 0.25,
  'visual_numeric': 0.2,
  'text_numeric': 0.05,
  'categorical_context': 0.05}}

## 7. Evaluacion rapida del retrieval

Esto recorre el indice y calcula metricas CBR. Puede tardar un poco.

In [12]:
from cbr_engine.evaluation import evaluate_index

metrics, per_query = evaluate_index(retriever, k_values=[5])
metrics

{'neighbor_outcome_correlation_pearson_at_5': 0.5494481741452774,
 'neighbor_outcome_correlation_spearman_at_5': 0.5304397294193764,
 'top_k_label_consistency_at_5': 0.5907407407407408,
 'hit_rate_top_performer_at_5': 0.018518518518518517,
 'fatigue_retrieval_consistency_at_5': 0.7490740740740741,
 'ndcg_at_5': 0.7710062913843533,
 'self_neighbor_violations': 0}

In [13]:
per_query.head()

,creative_id,k,actual,neighbor_prediction,self_in_neighbors,label_consistency,hit_top_performer,ndcg,fatigue_consistency
0,500000,5,0.341049,0.495350,False,0.8,0.0,0.955830,0.8
1,500001,5,0.597840,0.688562,False,0.4,0.0,0.570642,0.4
2,500002,5,0.607407,0.591848,False,0.6,0.0,0.967468,0.6
3,500003,5,0.491667,0.532080,False,0.8,0.0,0.982892,0.8
4,500004,5,0.519136,0.669561,False,0.4,0.0,0.501266,0.6


## 8. Opcional: guardar trace de la query

In [14]:
from cbr_engine.reporting import write_query_trace

trace_to_save = dict(trace)
trace_to_save.update(enriched)

RESULTS_DIR = ROOT / "outputs" / "cbr_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
write_query_trace(RESULTS_DIR / f"query_{creative_id}_notebook.json", trace_to_save)

RESULTS_DIR / f"query_{creative_id}_notebook.json"

WindowsPath('c:/Users/sylbo/Documents_local/hack_UPC_2026/outputs/cbr_results/query_500000_notebook.json')